# 07 — Figures for the QTTA study (STE vs Gaussian-smoothed surrogate gradients)
Put `scan.csv`, `scan_weights.csv`, `oracle_15c.csv` in DATA_DIR. Figures are saved to OUT_DIR.

In [ ]:
import os, numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import NullFormatter
DATA_DIR = os.environ.get("DATA_DIR", ".")
OUT_DIR = os.environ.get("OUT_DIR", ".")
plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
rng = np.random.default_rng(0)
def boot(v, B=5000):
    v = np.asarray(v, float); b = [v[rng.integers(0, len(v), len(v))].mean() for _ in range(B)]
    return v.mean(), np.percentile(b, 2.5), np.percentile(b, 97.5)
orc = pd.read_csv(f"{DATA_DIR}/oracle_15c.csv")
scan = pd.read_csv(f"{DATA_DIR}/scan.csv")
wts = pd.read_csv(f"{DATA_DIR}/scan_weights.csv")

## Fig 1 — Gaussian minus STE vs sigma/Delta (15 corruptions, paired, 95% bootstrap CI over corruptions)

In [ ]:
SIG = {"g0.25": 0.25, "g0.35": 0.35, "g0.5": 0.5, "g1.0": 1.0, "g2.0": 2.0}
fig, ax = plt.subplots(figsize=(5.2, 3.4))
for mn, col in [("W4A4", "C0"), ("W3A3", "C3")]:
    M = orc[orc.model == mn].pivot_table(index="corruption", columns="method", values="acc")
    xs, ys, lo, hi = [], [], [], []
    for k, s in SIG.items():
        m, l, h = boot(M[k] - M["STE"]); xs.append(s); ys.append(m); lo.append(m - l); hi.append(h - m)
    ax.errorbar(xs, ys, yerr=[lo, hi], marker="o", capsize=3, color=col, label=mn)
ax.axhline(0, color="k", lw=0.8); ax.set_xscale("log"); ax.set_xticks(list(SIG.values())); ax.set_xticklabels(list(map(str, SIG.values()))); ax.xaxis.set_minor_formatter(NullFormatter())
ax.set_xlabel(r"$\sigma/\Delta$"); ax.set_ylabel("accuracy vs STE (points)"); ax.set_title("Gaussian surrogate vs STE, 15 corruptions"); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig1_sigma_curve.png"); plt.show()

## Fig 2 — Gradient similarity to STE and TTA ceiling (values copied from notebooks 02 and 04 printouts)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3.4))
sg = [0.1, 0.25, 0.5, 1.0, 2.0]; cs = [-0.084, 0.804, 0.988, 0.966, 0.918]
axs[0].plot(sg, cs, "o-"); axs[0].axhline(1, color="gray", lw=0.6, ls="--"); axs[0].set_xscale("log")
axs[0].set_xticks(sg); axs[0].set_xticklabels(map(str, sg)); axs[0].xaxis.set_minor_formatter(NullFormatter()); axs[0].set_xlabel(r"$\sigma/\Delta$"); axs[0].set_ylabel("cosine(g_gauss, g_STE)")
axs[0].set_title("BN-gradient similarity to STE (W4A4, one batch)")
mods = ["FP32", "W8A8", "W4A4", "W4A4-minmax", "W3A3"]
no = [35.24, 35.27, 27.81, 31.46, 14.19]; bn = [72.51, 72.50, 70.46, 67.24, 64.98]; st = [76.74, 76.82, 74.82, 71.64, 69.52]
x = np.arange(len(mods)); w = 0.27
for i, (v, l) in enumerate([(no, "no TTA"), (bn, "BN stats only"), (st, "BN + entropy (STE)")]): axs[1].bar(x + (i - 1) * w, v, w, label=l)
axs[1].set_xticks(x); axs[1].set_xticklabels(mods, rotation=20); axs[1].set_ylabel("mean accuracy, 3 corruptions (%)"); axs[1].legend(frameon=False, fontsize=8)
axs[1].set_title("TTA gain is ~4.2-4.5 pts regardless of quantizer")
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig2_gradient_and_ceiling.png"); plt.show()

## Fig 3 — Optimizer-stress scan: Gaussian minus STE (BN-only; lr x batch size)

In [ ]:
T = scan.groupby(["model", "lr", "bs", "method"]).acc.mean().unstack("method")
D = T[["g0.35", "g1.0", "g2.0"]].sub(T["STE"], axis=0)
labels = [f"{m}  lr={lr:g} bs={bs}" for m, lr, bs in D.index]
fig, ax = plt.subplots(figsize=(5.2, 4.6))
im = ax.imshow(D.values, cmap="RdBu", vmin=-6, vmax=6, aspect="auto")
ax.set_xticks(range(3)); ax.set_xticklabels([r"$\sigma/\Delta$=0.35", "1.0", "2.0"]); ax.set_yticks(range(len(D))); ax.set_yticklabels(labels, fontsize=8)
for i in range(D.shape[0]):
    for j in range(D.shape[1]): ax.text(j, i, f"{D.values[i, j]:+.1f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, label="points vs STE"); ax.set_title("Stress scan (3 corruptions, 2 seeds)")
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig3_stress_scan.png"); plt.show()

## Fig 4 — Adapting weights too: Gaussian minus STE

In [ ]:
w2 = wts[wts.group != "BN"]; ref = wts[wts.group == "BN"].groupby("model").acc.mean()
T = w2.groupby(["model", "group", "lr_w", "method"]).acc.mean().unstack("method")
D = T[["g0.25", "g0.5", "g1.0"]].sub(T["STE"], axis=0)
labels = [f"{m} {g} lr_w={lw:g}\n(STE {T.loc[(m, g, lw), 'STE']:.1f} vs BN-only {ref[m]:.1f})" for m, g, lw in D.index]
fig, ax = plt.subplots(figsize=(5.6, 5.0))
im = ax.imshow(D.values, cmap="RdBu", vmin=-4, vmax=4, aspect="auto")
ax.set_xticks(range(3)); ax.set_xticklabels([r"$\sigma/\Delta$=0.25", "0.5", "1.0"]); ax.set_yticks(range(len(D))); ax.set_yticklabels(labels, fontsize=7)
for i in range(D.shape[0]):
    for j in range(D.shape[1]): ax.text(j, i, f"{D.values[i, j]:+.1f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, label="points vs STE"); ax.set_title("BN + weight adaptation")
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig4_weight_adaptation.png"); plt.show()

## Fig 5 — Oracle test: ceiling for any corruption-aware sigma (leave-one-seed-out selection, 95% CI over corruptions)

In [ ]:
G = ["g0.25", "g0.35", "g0.5", "g1.0", "g2.0"]; res = {}
for mn in ["W4A4", "W3A3"]:
    A = orc[orc.model == mn].pivot_table(index=["corruption", "seed"], columns="method", values="acc")
    CORR = sorted(orc.corruption.unique()); seeds = sorted(orc.seed.unique())
    r_or, r_fix, r_ste = ({c: [] for c in CORR} for _ in range(3))
    for s in seeds:
        sel = A.xs(s, level="seed"); ev = A.drop(s, level="seed").groupby("corruption").mean(); fixed = sel[G].mean().idxmax()
        for c in CORR:
            r_or[c].append(ev.loc[c, sel.loc[c, G].idxmax()]); r_fix[c].append(ev.loc[c, fixed]); r_ste[c].append(ev.loc[c, "STE"])
    f = lambda d: np.array([np.mean(d[c]) for c in CORR])
    res[mn] = (boot(f(r_fix) - f(r_ste)), boot(f(r_or) - f(r_fix)))
    print(mn, "best fixed - STE: %.2f [%.2f, %.2f] | oracle - best fixed: %.2f [%.2f, %.2f]" % (*res[mn][0], *res[mn][1]))
fig, ax = plt.subplots(figsize=(5.2, 3.4)); x = np.arange(2); w = 0.35
for i, mn in enumerate(res):
    for k, (m, l, h) in enumerate(res[mn]):
        ax.errorbar(k + (i - 0.5) * w, m, yerr=[[m - l], [h - m]], fmt="o", capsize=4, color=["C0", "C3"][i], label=mn if k == 0 else None)
ax.axhline(0, color="k", lw=0.8); ax.axhline(0.3, color="gray", ls="--", lw=0.8); ax.text(1.45, 0.31, "build threshold (0.3)", ha="right", fontsize=8, color="gray")
ax.set_xticks(x); ax.set_xticklabels(["best fixed $\\sigma$\nminus STE", "per-corruption oracle\nminus best fixed"]); ax.set_xlim(-0.6, 1.6)
ax.set_ylabel("points (95% CI)"); ax.set_title("Headroom for adaptive sigma"); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig5_oracle.png"); plt.show()